# Portfolio Optimization Through Divi

This notebook demonstrates portfolio optimization using quantum algorithms, specifically the Quantum Approximate Optimization Algorithm (QAOA) via the Divi framework. We use spectral partitioning to break down large portfolios into smaller, more tractable sub-problems that can be solved on quantum hardware.

**Approach:**
1. Partition assets into correlated clusters using spectral community detection
2. Build QUBO (Quadratic Unconstrained Binary Optimization) formulations for each partition
3. Solve each partition independently using QAOA
4. Evaluate solutions against classical optimal benchmarks

**Dataset:** Financial data from 2016-01-01

## Data Preparation

We load three key matrices that define our portfolio optimization problem:

- **Returns Matrix** (`full_returns`): Expected returns for each asset
- **Correlation Matrix** (`correlation_matrix`): Pairwise correlations between assets (normalized to [-1, 1])
- **Covariance Matrix** (`covariance_matrix`): Pairwise covariances, representing risk relationships

The correlation matrix captures normalized relationships, while the covariance matrix quantifies actual risk magnitudes. Both are essential: correlation for partitioning (identifying asset clusters), and covariance for optimization (quantifying portfolio risk).

In [ ]:
import numpy as np

In [ ]:
full_returns = np.load("2016-01-01_returns.npy")
correlation_matrix = np.load("2016-01-01_correlation.npy")
covariance_matrix = np.load("2016-01-01_covariance.npy")

### Data Scaling

Quantum algorithms work best with values in a reasonable range (typically around 0-1 or -1 to 1). Our original data has very small values (~0.0001), so we apply a simple multiplicative scaling factor of 10,000.

**Why multiplicative scaling?**
- Preserves all financial relationships (ratios, correlations, relative magnitudes)
- Maintains the mathematical structure needed for portfolio optimization
- Makes values more suitable for quantum gate operations

**Why not scale the correlation matrix?**
The correlation matrix is already normalized to [-1, 1] and represents relationships between assets. Scaling would distort these relationships and break the normalization.

In [ ]:
# Values are ~0.0001, so ×10,000 brings them to ~1.0 (better for quantum gates)
scale_factor = 10000

scaled_full_returns = full_returns * scale_factor
scaled_covariance_matrix = covariance_matrix * scale_factor

# Don't scale correlation matrix - it's already normalized to [-1, 1] range
# and represents relationships between assets. Scaling would distort these relationships.
scaled_correlation_matrix = correlation_matrix

## Partitioning Data

Large portfolios create QUBO problems that are too large for current quantum hardware. We use **spectral partitioning** to identify clusters of correlated assets, then solve each cluster independently.

**Benefits:**
- Reduces problem size: Each partition can be solved separately
- Maintains structure: Correlated assets stay together
- Enables quantum advantage: Smaller problems fit on quantum hardware
- Preserves optimization quality: Well-separated clusters can be optimized independently

The partitioning algorithm uses the correlation matrix to identify communities of assets that are more correlated within clusters than between clusters.

In [ ]:
from modularity_spectral_partitioning import modularity_spectral_threshold
from visualization import (
    plot_reordered_correlation,
    plot_partition_counts,
    evaluate_partition_quality,
    print_partition_quality,
    sweep_partition_thresholds,
    plot_partition_sweep_results,
)

### Finding the Optimal Partition Threshold

We need to choose a threshold that determines the maximum size of each partition. Too small → too many partitions (overhead). Too large → problems still too big for quantum solvers.

**Metrics to consider:**
- **Modularity**: Measures how well the partition captures community structure (higher is better, typically >0.3 is good)
- **Separation Ratio**: Ratio of intra-cluster to inter-cluster correlations (higher is better, >1.5 is useful for portfolio optimization)

Let's sweep through different threshold values to find the best balance:

In [ ]:
sweep_results = sweep_partition_thresholds(
    correlation_matrix,
    min_threshold=10,
    max_threshold=30,
    step=5,
)

# Visualize the sweep results
plot_partition_sweep_results(sweep_results)


### Selected Partition Threshold

We chose a threshold of **20 assets per partition** based on the sweep results. This provides:
- Good separation ratio (~1.7-1.8) for meaningful cluster independence
- Reasonable partition sizes that fit within quantum hardware constraints
- Balanced trade-off between modularity and separation

**What this means:** No partition will contain more than 20 assets, ensuring each sub-problem is small enough for quantum solvers while maintaining meaningful asset relationships.

In [ ]:
MAX_PARTITION_SIZE = 20

### Visualizing Partition Structure

The reordered correlation matrix shows how assets cluster together. Look for:
- **Strong colors within diagonal blocks**: High intra-cluster correlations (good)
- **Weak colors between blocks**: Low inter-cluster correlations (good)
- **Clear boundaries**: Well-separated clusters make independent optimization valid

In [ ]:
communities, partitions = modularity_spectral_threshold(correlation_matrix, threshold=MAX_PARTITION_SIZE)
plot_partition_counts(correlation_matrix, partitions, threshold=MAX_PARTITION_SIZE)

In [ ]:
# Visualize correlation matrix REORDERED by partitions to see clusters clearly
plot_reordered_correlation(correlation_matrix, partitions)

### Partition Quality Assessment

We evaluate partition quality using several metrics:

- **Intra-Cluster Correlation ↑**: How correlated assets are within clusters (higher = better clustering)
- **Inter-Cluster Correlation ↓**: How correlated clusters are with each other (lower = better separation)
- **Separation Ratio ↑**: Intra/Inter ratio (higher = clusters are more independent)
- **Modularity ↑**: Community structure quality (higher = better community detection)
- **CV ↓**: Coefficient of variation of cluster sizes (lower = more balanced partitions)

**Interpretation:**
- Separation Ratio > 1.5: Good for portfolio optimization
- Modularity > 0.3: Strong community structure (but financial data often has lower modularity)
- CV < 0.3: Well-balanced partition sizes

In [ ]:
# Quantify partition quality
quality_metrics = evaluate_partition_quality(correlation_matrix, partitions)
print_partition_quality(quality_metrics)

### Creating Independent Sub-Problems

For each partition, we extract:
- **Sub-covariance matrix**: Risk relationships within the cluster
- **Sub-returns vector**: Expected returns for assets in the cluster

Each partition becomes an independent portfolio optimization problem that can be solved separately. This is valid because:
1. Clusters are well-separated (low inter-cluster correlation)
2. Assets within clusters are more correlated than across clusters
3. The global solution can be reconstructed from partition solutions

In [ ]:
# Extract sub-problems for each cluster
partitioned_covariance_matrices = {}
partitioned_returns = {}

for cluster_label in np.unique(partitions):
    cluster_indices = np.flatnonzero(partitions == cluster_label)
    partitioned_covariance_matrices[cluster_label] = scaled_covariance_matrix[
        np.ix_(cluster_indices, cluster_indices)
    ]
    partitioned_returns[cluster_label] = scaled_full_returns[cluster_indices]

## Solving Using Divi's Classes

In [ ]:
from visualization import analyze_lambda_selection
from utils import build_qubo_matrices_for_all_partitions

In [ ]:
# Analyze risk-return scale to guide lambda selection
analyze_lambda_selection(scaled_covariance_matrix, scaled_full_returns)

#### Understanding Lambda (λ)

The **lambda parameter** controls the risk-return trade-off in our QUBO formulation:

- **Low λ (conservative)**: Prioritizes risk minimization → safer portfolios
- **Medium λ (balanced)**: Equal weight on risk and return
- **High λ (aggressive)**: Prioritizes return maximization → riskier portfolios

**Our choice (λ = 0.75):**
Slightly aggressive, meaning we're willing to accept moderate risk for better returns. This is above the "balanced" value of 0.51, indicating a preference for return over pure risk minimization.

In [ ]:
LAMBDA_PARAM = 0.75

In [ ]:
# Build QUBO matrices for all partitions once (with lambda = 0.75)
# This ensures all solvers work on the exact same problem formulation
qubo_matrices = build_qubo_matrices_for_all_partitions(
    partitioned_returns_dict=partitioned_returns,
    partitioned_covariance_dict=partitioned_covariance_matrices,
    lambda_param=LAMBDA_PARAM,
)

print(f"Built QUBO matrices for {len(qubo_matrices)} partitions")

### Single-instance QAOA Example

**Divi** is a quantum programming framework that provides a high-level interface for QAOA. It handles:
- Quantum circuit construction
- Parameter optimization
- Backend management (simulators or real hardware)
- Solution extraction

We use Divi's `QAOA` class with:
- **MonteCarloOptimizer**: Population-based optimization for parameter tuning
- **ParallelSimulator**: Quantum circuit simulation backend
- **Configurable depth**: Number of QAOA layers (more layers = better approximation, but slower)

In [ ]:
from divi.qprog import QAOA
from divi.qprog.optimizers import MonteCarloOptimizer
from divi.backends import ParallelSimulator
from utils import evaluate_solution

#### Example Partition Analysis

We'll demonstrate the QAOA solution process on **Partition 18** as an example. This partition contains assets that are highly correlated with each other, making it a good candidate for independent optimization.

In [ ]:
partition_id = 18

In [ ]:
qaoa_problem = QAOA(
    problem=qubo_matrices[partition_id],
    n_layers=2,
    # optimizer=ScipyOptimizer(method=ScipyMethod.COBYLA),
    optimizer=MonteCarloOptimizer(population_size=20, n_best_sets=5, keep_best_params=True),
    max_iterations=15,
    backend=ParallelSimulator(shots=10000)
)

qaoa_problem.run()

divi_solution = qaoa_problem.solution

#### Solution Evaluation Function

This function compares QAOA solutions to the optimal classical solution (found using exact enumeration). It reports:

**QUBO Energy Metrics:**
- How close the QAOA solution is to the optimal QUBO energy
- Whether QAOA found the exact optimal solution

**Financial Metrics:**
- **Return**: Expected portfolio return (μ^T x)
- **Risk**: Portfolio variance (x^T Σ x)
- **Sharpe Ratio**: Return/Risk ratio (higher is better)
- **Number of Assets**: How many assets are selected

This comparison helps us understand:
1. How well QAOA performs compared to classical methods
2. Whether the QUBO formulation captures the financial objectives
3. The practical impact of suboptimal solutions

In [ ]:
evaluate_solution(
    qubo_matrix=qubo_matrices[partition_id],
    qaoa_solution=divi_solution,
    returns=partitioned_returns[partition_id],
    covariance_matrix=partitioned_covariance_matrices[partition_id],
    partition_id=partition_id,
)

### Solving Multiple QAOAs through `ProgramBatch` Class

Divi provides the infrastructure to instantiate multiple Quantum Programs and orchestrate their orchestration and the ability to provide the technique  with which to aggregate their respective results according to the user's needs.

In the `qubo_batch.py` file, one can see a concrete implementation of such a class. The main outline of any implementation of `ProgramBatch` needs to only provide two main functions:
- `create_programs`: This instantiates all the instances of `QuantumProgram` (the parent class of `QAOA`).
- `aggregate_results`: This collects the results from all the programs and aggregates them.

Take a moment to understand the basic structure of such classes by checking the code in the file.


In [ ]:
from qubo_batch import QUBOBatch
from utils import solve_and_aggregate_partitions, compare_portfolio_solutions
from divi.backends import QoroService

In [ ]:
batch = QUBOBatch(
    qubos=qubo_matrices,
    n_layers=2,
    max_iterations=10,
    partitions=partitions,
    backend=ParallelSimulator(), # Comment this out to use QoroService
    # backend = QoroService() # Uncomment to use QoroService if you have an API key
)

# Create programs, run, and aggregate
batch.create_programs()
batch.run(blocking=True)
qaoa_bitstring = batch.aggregate_results()

In [ ]:
exact_bitstring = solve_and_aggregate_partitions(qubo_matrices, partitions)

In [ ]:
compare_portfolio_solutions(
    qaoa_bitstring,
    exact_bitstring,
    scaled_full_returns,
    scaled_covariance_matrix,
)